# Learning 01: LLM-Driven NER Dataset Generation

**Domain**: Cybersecurity threat intelligence

Training a GLiNER model from scratch requires annotated data in the format:
```json
{"tokenized_text": ["The", "LockBit", ...], "ner": [[1, 2, "malware"], ...]}
```
Manually annotating hundreds of texts is expensive. LLMs can automate this at scale.

## Workflow
1. Feed raw text to LLM → receive `entity_text <> entity_type` lines
2. Tokenize text with the same tokenizer GLiNER uses
3. Find token-level spans for each extracted entity
4. Build `{tokenized_text, ner}` training examples

In [ ]:
import re, json, os
from openai import OpenAI

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "raw_security_reports.json")) as f:
    reports = json.load(f)

api_key = "your-api-key-here"  # Replace with your actual OpenAI key
client = OpenAI(api_key=api_key)

print(f"✓ Loaded {len(reports)} raw security reports")
print(f"\nSample report:")
print(reports[0]['text'])

## 1. Tokenizer

GLiNER uses word-level tokenization (not BPE). We use the same regex to align LLM output with training format.

In [2]:
def tokenize(text):
    """GLiNER-compatible word tokenizer."""
    return re.findall(r'\w+(?:[-_]\w+)*|\S', text)


text = reports[0]['text']
tokens = tokenize(text)
print(f"Text: {text[:80]}...")
print(f"Tokens (first 20): {tokens[:20]}")
print(f"Total tokens: {len(tokens)}")
print()
# Note: '3.0' becomes ['3', '.', '0'] — punctuation is split
print("Note: '3.0' → ", tokenize("3.0"))
print("Note: 'CVE-2023-4966' → ", tokenize("CVE-2023-4966"))  # hyphenated: stays as 1 token

Text: The LockBit 3.0 ransomware group exploited CVE-2023-4966 in Citrix NetScaler ADC...
Tokens (first 20): ['The', 'LockBit', '3', '.', '0', 'ransomware', 'group', 'exploited', 'CVE-2023-4966', 'in', 'Citrix', 'NetScaler', 'ADC', 'to', 'gain', 'initial', 'access', '.', 'After', 'lateral']
Total tokens: 43

Note: '3.0' →  ['3', '.', '0']
Note: 'CVE-2023-4966' →  ['CVE-2023-4966']


## 2. LLM Annotation Prompt

Entity types for cybersecurity NER:
| Type | Examples |
|------|----------|
| `malware` | LockBit, Emotet, SUNBURST, TrickBot |
| `vulnerability` | CVE-2023-4966, Log4Shell, ProxyNotShell |
| `attack_technique` | phishing, SQL injection, lateral movement |
| `affected_software` | Citrix NetScaler, Windows Server, Apache Log4j |
| `threat_actor` | APT29, Lazarus Group, FIN7 |

In [3]:
ENTITY_TYPES = ["malware", "vulnerability", "attack_technique", "affected_software", "threat_actor"]

SYSTEM_PROMPT = """Extract named entities from cybersecurity texts.
Output one entity per line in format: entity_text <> entity_type
Use ONLY these entity types: malware, vulnerability, attack_technique, affected_software, threat_actor
Output ONLY entity lines, nothing else. Do not include explanations or headers."""


def call_llm_ner(client, text, model="gpt-4o-mini"):
    """Call LLM to extract entities. Returns raw response text."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text}
        ],
        temperature=0  # Deterministic for annotation
    )
    return response.choices[0].message.content


raw_output = call_llm_ner(client, reports[0]['text'])
print("LLM output:")
print(raw_output)

LLM output:
LockBit 3.0 <> threat_actor  
CVE-2023-4966 <> vulnerability  
Citrix NetScaler ADC <> affected_software  
Windows Server <> affected_software  
VMware ESXi <> affected_software  


## 3. Parse LLM Output

In [4]:
def parse_ner_output(raw_output):
    """Parse 'entity_text <> entity_type' lines into (text, type) pairs."""
    entities = []
    for line in raw_output.strip().split('\n'):
        line = line.strip()
        if '<>' not in line:
            continue
        parts = line.split('<>')
        if len(parts) == 2:
            entity_text = parts[0].strip()
            entity_type = parts[1].strip().lower()
            if entity_text and entity_type in ENTITY_TYPES:
                entities.append((entity_text, entity_type))
    return entities


entities = parse_ner_output(raw_output)
print(f"Parsed {len(entities)} entities:")
for text_ent, ent_type in entities:
    print(f"  [{ent_type}] '{text_ent}'")

Parsed 5 entities:
  [threat_actor] 'LockBit 3.0'
  [vulnerability] 'CVE-2023-4966'
  [affected_software] 'Citrix NetScaler ADC'
  [affected_software] 'Windows Server'
  [affected_software] 'VMware ESXi'


## 4. Find Token Spans

GLiNER needs `[start_token_idx, end_token_idx, label]` (inclusive, 0-indexed).

In [5]:
def find_token_span(tokenized_text, entity_text):
    """
    Find the token-level span of entity_text in tokenized_text.
    Returns (start_idx, end_idx) inclusive, or None if not found.
    """
    entity_tokens = tokenize(entity_text)
    n = len(entity_tokens)
    for i in range(len(tokenized_text) - n + 1):
        window = [t.lower() for t in tokenized_text[i:i + n]]
        if window == [t.lower() for t in entity_tokens]:
            return (i, i + n - 1)  # inclusive end index
    return None


text = reports[0]['text']
tokens = tokenize(text)

print(f"Token alignment:")
for entity_text, entity_type in entities:
    span = find_token_span(tokens, entity_text)
    if span:
        s, e = span
        reconstructed = ' '.join(tokens[s:e+1])
        print(f"  [{entity_type}] span=[{s},{e}] → '{reconstructed}'")
    else:
        print(f"  [{entity_type}] '{entity_text}' → NOT FOUND (will be skipped)")

Token alignment:
  [threat_actor] span=[1,4] → 'LockBit 3 . 0'
  [vulnerability] span=[8,8] → 'CVE-2023-4966'
  [affected_software] span=[10,12] → 'Citrix NetScaler ADC'
  [affected_software] span=[36,37] → 'Windows Server'
  [affected_software] span=[39,40] → 'VMware ESXi'


## 5. Full Pipeline: Build GLiNER Training Example

In [6]:
def build_gliner_example(client, text):
    """
    Annotate a single text using LLM and build a GLiNER training example.
    
    Returns:
        dict with 'tokenized_text' and 'ner' keys, or None if no entities found
    """
    tokenized_text = tokenize(text)
    raw_output = call_llm_ner(client, text)
    entities = parse_ner_output(raw_output)
    
    ner_spans = []
    for entity_text, entity_type in entities:
        span = find_token_span(tokenized_text, entity_text)
        if span:
            ner_spans.append([span[0], span[1], entity_type])
    
    if not ner_spans:
        return None
    return {"tokenized_text": tokenized_text, "ner": ner_spans}


example = build_gliner_example(client, reports[2]['text'])
print("GLiNER training example:")
print(json.dumps(example, indent=2))

GLiNER training example:
{
  "tokenized_text": [
    "APT29",
    ",",
    "also",
    "known",
    "as",
    "Cozy",
    "Bear",
    ",",
    "leveraged",
    "a",
    "zero-day",
    "vulnerability",
    "CVE-2024-1234",
    "in",
    "Microsoft",
    "Exchange",
    "Server",
    "to",
    "install",
    "the",
    "SUNBURST",
    "backdoor",
    "on",
    "government",
    "networks",
    ".",
    "The",
    "group",
    "used",
    "living-off-the-land",
    "techniques",
    "to",
    "avoid",
    "detection",
    "."
  ],
  "ner": [
    [
      12,
      12,
      "vulnerability"
    ],
    [
      14,
      16,
      "affected_software"
    ],
    [
      20,
      20,
      "malware"
    ],
    [
      0,
      0,
      "threat_actor"
    ],
    [
      29,
      29,
      "attack_technique"
    ]
  ]
}


## 6. Batch Annotation and Validation

In [7]:
from tqdm import tqdm


def build_ner_dataset(client, reports, max_examples=None):
    """Build a complete GLiNER training dataset from raw reports."""
    dataset = []
    texts = [r['text'] for r in reports]
    if max_examples:
        texts = texts[:max_examples]
    
    for text in tqdm(texts, desc="Annotating NER"):
        try:
            example = build_gliner_example(client, text)
            if example:
                dataset.append(example)
        except Exception as e:
            print(f"  Error: {e}")
    return dataset


ner_dataset = build_ner_dataset(client, reports, max_examples=5)

print(f"\n✓ Generated {len(ner_dataset)} NER examples")
print("\nDataset stats:")
from collections import Counter
all_entity_types = [span[2] for ex in ner_dataset for span in ex['ner']]
for ent_type, count in Counter(all_entity_types).most_common():
    print(f"  {ent_type:20}: {count}")

Annotating NER:   0%|          | 0/5 [00:00<?, ?it/s]

Annotating NER:  20%|██        | 1/5 [00:01<00:07,  1.77s/it]

Annotating NER:  40%|████      | 2/5 [00:02<00:04,  1.45s/it]

Annotating NER:  60%|██████    | 3/5 [00:04<00:02,  1.47s/it]

Annotating NER:  80%|████████  | 4/5 [00:05<00:01,  1.34s/it]

Annotating NER: 100%|██████████| 5/5 [00:06<00:00,  1.08s/it]

Annotating NER: 100%|██████████| 5/5 [00:06<00:00,  1.25s/it]


✓ Generated 5 NER examples

Dataset stats:
  affected_software   : 6
  threat_actor        : 3
  attack_technique    : 3
  malware             : 3
  vulnerability       : 2


In [8]:
# Validate dataset: all spans must be within bounds
def validate_ner_dataset(dataset):
    errors = []
    for i, ex in enumerate(dataset):
        n_tokens = len(ex['tokenized_text'])
        for span in ex['ner']:
            s, e, label = span
            if not (0 <= s <= e < n_tokens):
                errors.append(f"Example {i}: invalid span [{s},{e}] for {n_tokens} tokens")
            if label not in ENTITY_TYPES:
                errors.append(f"Example {i}: unknown entity type '{label}'")
    return errors


errors = validate_ner_dataset(ner_dataset)
if errors:
    print(f"❌ Validation errors: {errors}")
else:
    print("✓ All spans valid")

# Save
output_path = os.path.join(fixtures, "..", "..", "fixtures", "expected", "ner_dataset_sample.json")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(os.path.normpath(output_path), 'w') as f:
    json.dump(ner_dataset, f, indent=2)
print(f"✓ Saved to ner_dataset_sample.json")

✓ All spans valid
✓ Saved to ner_dataset_sample.json
